<a href="https://colab.research.google.com/github/Aldo-Idk04/Colab-Tabla-de-Simbolos/blob/main/Mi_Copia_de_Pr%C3%A1ctica_Tabla_de_s%C3%ADmbolos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programación de Sistemas de Base II - Práctica: Elementos de la Programación de Sistemas, Máquinas de Pila y Ámbitos de Tabla de Símbolos
  
**Asignatura:** Programación de Sistemas de Base II (9.º Semestre)  
**Unidad II:** Elementos de la Programación de Sistemas  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

## 1. Objetivos de Aprendizaje    
1. Simular el comportamiento de una máquina de pila abstracta para la evaluación y generación de instrucciones.  
2. Construir e inspeccionar una Tabla de Símbolos avanzada con soporte para bloques anidados y manejo de ámbitos (Scopes).  
3. Comprender el mecanismo de traducción de expresiones simples y el rol de los ensambladores y cargadores en la cadena de desarrollo de sistemas.

## 2. Conexión Teórica (NotebookLM)    
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:    
> - *¿Qué diferencia a un ensamblador de una pasada de uno de dos pasadas?*  
>   
>   **Respuesta:**  
>   La diferencia fundamental recae en cuántas veces el ensamblador lee el código fuente antes de producir el archivo objeto final.  
>   * **Ensamblador de una sola pasada (Single-Pass Assembler):** Lee y procesa el archivo fuente una sola vez. A medida que lee cada línea, traduce los mnemónicos a código máquina e incrementa el contador de direcciones.  
>     * *El problema de las referencias hacia adelante (Forward References):* Si una instrucción salta a una etiqueta que se define más adelante en el archivo (por ejemplo, `JMP FIN` antes de declarar la etiqueta `FIN:`), el ensamblador no conoce la dirección de memoria de `FIN` en ese instante.  
>     * *Cómo lo resuelve:* Utiliza una Tabla de Parches o Correcciones (*Fixup Table*). Deja la dirección del salto como pendiente o en blanco y, cuando finalmente encuentra la etiqueta en una línea posterior, regresa a rellenar (*patching/backpatching*) las instrucciones que quedaron inconclusas.  
>   * **Ensamblador de dos pasadas (Two-Pass Assembler):** Realiza dos barridos independientes sobre el código fuente para resolver la memoria limpiamente:  
>     * *Primera pasada (Mapeo de símbolos):* Recorre el código únicamente para contabilizar el tamaño de las instrucciones mediante el Contador de Localidades (*Location Counter* - LC) y registrar todas las etiquetas en la Tabla de Símbolos (TABSIM) asociándolas a su dirección de memoria correspondiente. En este paso no genera código máquina.  
>     * *Segunda pasada (Traducción):* Vuelve a leer el archivo desde el principio. Como la Tabla de Símbolos ya está completa, sustituye cada mnemónico y etiqueta por su valor binario correspondiente sin problemas de referencias pendientes.
>
> - *¿Cómo evalúa una máquina de pila la expresión posfija `a b + c *`?*  
>   
>   **Respuesta:**  
>   Una máquina de pila evalúa la Notación Polaca Inversa (posfija) de izquierda a derecha siguiendo dos reglas operativas simples:  
>   1. Si el elemento es un operando (variable o constante), se coloca en la cima de la pila mediante la operación `PUSH`.  
>   2. Si el elemento es un operador binario, se extraen los dos elementos del tope mediante la operación `POP`, se ejecuta la operación entre ellos y el resultado se regresa a la cima con un `PUSH`.  
>   
>   **Paso a paso de la evaluación para `a b + c *`:**  
>   * **Leer `a`:** Es un operando $\rightarrow$ hace `PUSH(a)`. Estado de la pila: `[ a ]`.  
>   * **Leer `b`:** Es un operando $\rightarrow$ hace `PUSH(b)`. Estado de la pila: `[ a, b ]` (donde $b$ está en la cima).  
>   * **Leer `+`:** Es un operador binario. Extrae los dos valores del tope: $\text{POP}() \to b$, $\text{POP}() \to a$. Realiza la suma: $T_1 = a + b$. Guarda el resultado intermedio: $\text{PUSH}(T_1)$. Estado de la pila: `[ (a + b) ]`.  
>   * **Leer `c`:** Es un operando $\rightarrow$ hace `PUSH(c)`. Estado de la pila: `[ (a + b), c ]` (donde $c$ está en la cima).  
>   * **Leer `*`:** Es un operador binario. Extrae los dos valores del tope: $\text{POP}() \to c$, $\text{POP}() \to (a + b)$. Realiza la multiplicación: $T_2 = (a + b) * c$. Guarda el resultado: $\text{PUSH}(T_2)$. Estado de la pila: `[ (a + b) * c ]`.  
>   
>   Al finalizar la lectura de los tokens, el resultado global de la expresión queda almacenado en la cima de la pila.
>
> - *¿Qué representa una entrada en la Tabla de Símbolos dentro de un entorno de ejecución?*  
>   
>   **Respuesta:**  
>   Una entrada en la Tabla de Símbolos es una estructura de datos que representa un registro individual asociado a cada identificador (variable, función, constante o tipo) declarado dentro del programa.  
>   Aunque se consulta constantemente en tiempo de compilación para validar reglas semánticas, en relación con el entorno de ejecución (*Run-Time Environment*), la entrada funciona como la asociación entre la abstracción del código fuente y el espacio físico o relativo en la memoria RAM.  
>   
>   **Componentes clave de una entrada:**  
>   * **Lexema / Nombre:** La cadena de texto original que identifica al símbolo (ej. `"suma"`, `"limite"`).  
>   * **Tipo de Dato:** Define la naturaleza de la información (`int`, `float`, `struct`, etc.), permitiendo determinar cuántos bytes ocupa en memoria.  
>   * **Dirección Relativa / Desplazamiento (Offset):** La posición de memoria asignada dentro del segmento de datos o dentro del Registro de Activación (*Stack Frame*) en la pila de ejecuciones.  
>   * **Categoría:** Especifica si se trata de una variable local, variable global, parámetro formal o identificador de función.  
>   * **Ámbito (Scope) y Anidamiento:** Permite resolver a qué bloque de código o función pertenece el identificador cuando existen variables con nombres idénticos en diferentes scopes.  
>   * **Valor (en intérpretes/entornos interactivos):** Cuando se trabaja con entornos de ejecución directa o intérpretes, la entrada puede contener directamente el valor actual asignado a la variable.


## 3. Sección Práctica 1: Simulación de una Máquina de Pila Abstracta (Stack Machine)  
Este código simula paso a paso la mecánica de un analizador ascendente (LR) con Pila Semántica y evaluación On-the-Fly.  
Se muestra el apilado/desapilado de tuplas semánticas y la emisión inmediata de Código de Tres Direcciones (3AC) conforme ocurren las reducciones, sin necesidad de construir un árbol sintáctico explícito en memoria.

### [CÓDIGO 1.1] Implementación del Emulador de Máquina de Pila
```python

class ElementoPila:
    """Estructura de la tupla/registro de la Pila Semántica: (estado, simbolo, atributo)"""
    def __init__(self, estado, simbolo, atributo=None):
        self.estado = estado
        self.simbolo = simbolo
        self.atributo = atributo

    def __repr__(self):
        val_str = f", attr={self.atributo}" if self.atributo is not None else ""
        return f"[{self.estado}: {self.simbolo}{val_str}]"


class SimulacionPilaSemantica:
    """
    Simula un analizador ascendente (LR) con Pila Semántica paralela
    que realiza la evaluación 'On-the-Fly' (al vuelo) generando Código de
    Tres Direcciones (3AC) sin construir un árbol sintáctico explícito en memoria.
    """
    def __init__(self):
        self.pila = [ElementoPila(0, '$')]  # Estado inicial 0 con centinela $
        self.contador_temporales = 0
        self.codigo_intermedio = []

    def nuevo_temporal(self):
        self.contador_temporales += 1
        return f"t{self.contador_temporales}"

    def emitir(self, instruccion):
        self.codigo_intermedio.append(instruccion)
        print(f"   ⚡ [ON-THE-FLY EMISIÓN]: {instruccion}")

    def mostrar_pila(self):
        pila_str = " ".join([str(elem) for elem in self.pila])
        print(f"PILA: {pila_str}")

    def desplazar(self, nuevo_estado, simbolo, atributo=None):
        """Operación Shift: Apila el estado, el símbolo y su atributo léxico."""
        elem = ElementoPila(nuevo_estado, simbolo, atributo)
        self.pila.append(elem)
        print(f"\n---> DESPLAZAR (Shift): Token '{simbolo}' (Atributo: {atributo}) -> Estado {nuevo_estado}")
        self.mostrar_pila()

    def reducir(self, num_regla, antecedente, num_simbolos_cuerpo, accion_semantica_fn, nuevo_estado_ir_a):
        """
        Operación Reduce:
        1. Valida que haya suficientes elementos para desapilar sin consumir el centinela $.
        2. Desapila los símbolos del cuerpo de la regla.
        3. Ejecuta la acción semántica On-the-Fly ($$).
        4. Apila el antecedente con su atributo sintetizado ($$) y el estado IR_A.
        """
        print(f"\n<--- REDUCIR (Reduce R{num_regla}): {antecedente} -> ({num_simbolos_cuerpo} símbolos)")

        if num_simbolos_cuerpo > len(self.pila) - 1:
            raise RuntimeError(f"Error de análisis: Pila insuficiente para reducir regla R{num_regla}.")

        elementos_extraidos = []
        for _ in range(num_simbolos_cuerpo):
            elementos_extraidos.insert(0, self.pila.pop())

        atributos_cuerpo = [elem.atributo for elem in elementos_extraidos]
        print(f"   Atributos desapilados ($1..${num_simbolos_cuerpo}): {atributos_cuerpo}")

        atributo_sintetizado = accion_semantica_fn(atributos_cuerpo)

        self.pila.append(ElementoPila(nuevo_estado_ir_a, antecedente, atributo_sintetizado))
        print(f"   Resultado sintetizado ($$ = {atributo_sintetizado}) apilado en estado {nuevo_estado_ir_a}")
        self.mostrar_pila()

    def aceptar(self):
        """Acción Accept: Valida que la pila contenga únicamente [$, E] al finalizar la entrada."""
        if len(self.pila) == 2 and self.pila[1].simbolo == 'E':
            print(f"\n✅ [ACCEPT]: Cadena reconocida exitosamente. Atributo final sintetizado en: {self.pila[1].atributo}")
            return True
        print("\n❌ [ERROR]: Estado de aceptación inválido en la pila.")
        return False


def transferir_atributo(attrs):
    """Acción semántica para reglas unitarias: $$ = $1"""
    return attrs[0]


if __name__ == "__main__":
    sim = SimulacionPilaSemantica()
    print("=" * 70)
    print("SIMULACIÓN DE LA PILA SEMÁNTICA Y TRADUCCIÓN ON-THE-FLY DE 'a + b * 5'")
    print("=" * 70)
    sim.mostrar_pila()

    # 1. Shift 'id' (a)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='a')

    # 2. Reduce (5) F -> id
    sim.reducir(num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=2)

    # 3. Reduce (4) T -> F
    sim.reducir(num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=1)

    # 4. Reduce (2) E -> T
    sim.reducir(num_regla=2, antecedente='E', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=4)

    # 5. Shift '+'
    sim.desplazar(nuevo_estado=6, simbolo='+')

    # 6. Shift 'id' (b)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='b')

    # 7. Reduce (5) F -> id
    sim.reducir(num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=2)

    # 8. Reduce (4) T -> F
    sim.reducir(num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=7)

    # 9. Shift '*'
    sim.desplazar(nuevo_estado=8, simbolo='*')

    # 10. Shift 'num' (5)
    sim.desplazar(nuevo_estado=5, simbolo='num', atributo='5')

    # 11. Reduce (6) F -> num
    sim.reducir(num_regla=6, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=9)

    # 12. Reduce (3) T -> T * F
    def accion_multiplicacion(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} * {attrs[2]}")
        return t

    sim.reducir(num_regla=3, antecedente='T', num_simbolos_cuerpo=3,
                accion_semantica_fn=accion_multiplicacion, nuevo_estado_ir_a=7)

    # 13. Reduce (1) E -> E + T
    def accion_suma(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} + {attrs[2]}")
        return t

    sim.reducir(num_regla=1, antecedente='E', num_simbolos_cuerpo=3,
                accion_semantica_fn=accion_suma, nuevo_estado_ir_a=4)

    # 14. Acción de Aceptación
    sim.aceptar()

    print("\n" + "=" * 70)
    print("CÓDIGO DE TRES DIRECCIONES GENERADO ON-THE-FLY:")
    print("=" * 70)
    for i, inst in enumerate(sim.codigo_intermedio, 1):
        print(f" {i}. {inst}")
    print("=" * 70)

```


In [ ]:

class ElementoPila:
    """Estructura de la tupla/registro de la Pila Semántica: (estado, simbolo, atributo)"""
    def __init__(self, estado, simbolo, atributo=None):
        self.estado = estado
        self.simbolo = simbolo
        self.atributo = atributo

    def __repr__(self):
        val_str = f", attr={self.atributo}" if self.atributo is not None else ""
        return f"[{self.estado}: {self.simbolo}{val_str}]"


class SimulacionPilaSemantica:
    """
    Simula un analizador ascendente (LR) con Pila Semántica paralela
    que realiza la evaluación 'On-the-Fly' (al vuelo) generando Código de
    Tres Direcciones (3AC) sin construir un árbol sintáctico explícito en memoria.
    """
    def __init__(self):
        self.pila = [ElementoPila(0, '$')]  # Estado inicial 0 con centinela $
        self.contador_temporales = 0
        self.codigo_intermedio = []

    def nuevo_temporal(self):
        self.contador_temporales += 1
        return f"t{self.contador_temporales}"

    def emitir(self, instruccion):
        self.codigo_intermedio.append(instruccion)
        print(f"   ⚡ [ON-THE-FLY EMISIÓN]: {instruccion}")

    def mostrar_pila(self):
        pila_str = " ".join([str(elem) for elem in self.pila])
        print(f"PILA: {pila_str}")

    def desplazar(self, nuevo_estado, simbolo, atributo=None):
        """Operación Shift: Apila el estado, el símbolo y su atributo léxico."""
        elem = ElementoPila(nuevo_estado, simbolo, atributo)
        self.pila.append(elem)
        print(f"\n---> DESPLAZAR (Shift): Token '{simbolo}' (Atributo: {atributo}) -> Estado {nuevo_estado}")
        self.mostrar_pila()

    def reducir(self, num_regla, antecedente, num_simbolos_cuerpo, accion_semantica_fn, nuevo_estado_ir_a):
        """
        Operación Reduce:
        1. Valida que haya suficientes elementos para desapilar sin consumir el centinela $.
        2. Desapila los símbolos del cuerpo de la regla.
        3. Ejecuta la acción semántica On-the-Fly ($$).
        4. Apila el antecedente con su atributo sintetizado ($$) y el estado IR_A.
        """
        print(f"\n<--- REDUCIR (Reduce R{num_regla}): {antecedente} -> ({num_simbolos_cuerpo} símbolos)")

        if num_simbolos_cuerpo > len(self.pila) - 1:
            raise RuntimeError(f"Error de análisis: Pila insuficiente para reducir regla R{num_regla}.")

        elementos_extraidos = []
        for _ in range(num_simbolos_cuerpo):
            elementos_extraidos.insert(0, self.pila.pop())

        atributos_cuerpo = [elem.atributo for elem in elementos_extraidos]
        print(f"   Atributos desapilados ($1..${num_simbolos_cuerpo}): {atributos_cuerpo}")

        atributo_sintetizado = accion_semantica_fn(atributos_cuerpo)

        self.pila.append(ElementoPila(nuevo_estado_ir_a, antecedente, atributo_sintetizado))
        print(f"   Resultado sintetizado ($$ = {atributo_sintetizado}) apilado en estado {nuevo_estado_ir_a}")
        self.mostrar_pila()

    def aceptar(self):
        """Acción Accept: Valida que la pila contenga únicamente [$, E] al finalizar la entrada."""
        if len(self.pila) == 2 and self.pila[1].simbolo == 'E':
            print(f"\n✅ [ACCEPT]: Cadena reconocida exitosamente. Atributo final sintetizado en: {self.pila[1].atributo}")
            return True
        print("\n❌ [ERROR]: Estado de aceptación inválido en la pila.")
        return False


def transferir_atributo(attrs):
    """Acción semántica para reglas unitarias: $$ = $1"""
    return attrs[0]


if __name__ == "__main__":
    sim = SimulacionPilaSemantica()
    print("=" * 70)
    print("SIMULACIÓN DE LA PILA SEMÁNTICA Y TRADUCCIÓN ON-THE-FLY DE 'a + b * 5'")
    print("=" * 70)
    sim.mostrar_pila()

    # 1. Shift 'id' (a)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='a')

    # 2. Reduce (5) F -> id
    sim.reducir(num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=2)

    # 3. Reduce (4) T -> F
    sim.reducir(num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=1)

    # 4. Reduce (2) E -> T
    sim.reducir(num_regla=2, antecedente='E', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=4)

    # 5. Shift '+'
    sim.desplazar(nuevo_estado=6, simbolo='+')

    # 6. Shift 'id' (b)
    sim.desplazar(nuevo_estado=3, simbolo='id', atributo='b')

    # 7. Reduce (5) F -> id
    sim.reducir(num_regla=5, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=2)

    # 8. Reduce (4) T -> F
    sim.reducir(num_regla=4, antecedente='T', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=7)

    # 9. Shift '*'
    sim.desplazar(nuevo_estado=8, simbolo='*')

    # 10. Shift 'num' (5)
    sim.desplazar(nuevo_estado=5, simbolo='num', atributo='5')

    # 11. Reduce (6) F -> num
    sim.reducir(num_regla=6, antecedente='F', num_simbolos_cuerpo=1,
                accion_semantica_fn=transferir_atributo, nuevo_estado_ir_a=9)

    # 12. Reduce (3) T -> T * F
    def accion_multiplicacion(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} * {attrs[2]}")
        return t

    sim.reducir(num_regla=3, antecedente='T', num_simbolos_cuerpo=3,
                accion_semantica_fn=accion_multiplicacion, nuevo_estado_ir_a=7)

    # 13. Reduce (1) E -> E + T
    def accion_suma(attrs):
        t = sim.nuevo_temporal()
        sim.emitir(f"{t} = {attrs[0]} + {attrs[2]}")
        return t

    sim.reducir(num_regla=1, antecedente='E', num_simbolos_cuerpo=3,
                accion_semantica_fn=accion_suma, nuevo_estado_ir_a=4)

    # 14. Acción de Aceptación
    sim.aceptar()

    print("\n" + "=" * 70)
    print("CÓDIGO DE TRES DIRECCIONES GENERADO ON-THE-FLY:")
    print("=" * 70)
    for i, inst in enumerate(sim.codigo_intermedio, 1):
        print(f" {i}. {inst}")
    print("=" * 70)


SIMULACIÓN DE LA PILA SEMÁNTICA Y TRADUCCIÓN ON-THE-FLY DE 'a + b * 5'
PILA: [0: $]

---> DESPLAZAR (Shift): Token 'id' (Atributo: a) -> Estado 3
PILA: [0: $] [3: id, attr=a]

<--- REDUCIR (Reduce R5): F -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 2
PILA: [0: $] [2: F, attr=a]

<--- REDUCIR (Reduce R4): T -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 1
PILA: [0: $] [1: T, attr=a]

<--- REDUCIR (Reduce R2): E -> (1 símbolos)
   Atributos desapilados ($1..$1): ['a']
   Resultado sintetizado ($$ = a) apilado en estado 4
PILA: [0: $] [4: E, attr=a]

---> DESPLAZAR (Shift): Token '+' (Atributo: None) -> Estado 6
PILA: [0: $] [4: E, attr=a] [6: +]

---> DESPLAZAR (Shift): Token 'id' (Atributo: b) -> Estado 3
PILA: [0: $] [4: E, attr=a] [6: +] [3: id, attr=b]

<--- REDUCIR (Reduce R5): F -> (1 símbolos)
   Atributos desapilados ($1..$1): ['b']
   Resultado sintetizad

## Simulación Avanzada de una Máquina de Pila Abstracta (Stack Machine)  
En esta sección implementaremos un intérprete/evaluador para una máquina de pila virtual, comprendiendo cómo las expresiones aritméticas compiladas se ejecutan a bajo nivel.  
## Objetivos de Aprendizaje    
Simular el comportamiento de una máquina de pila abstracta para la evaluación y generación de instrucciones de tipo **PUSH/POP/EVAL**.

### [CÓDIGO 1.2] Implementación del Emulador de Máquina de Pila
```python

class MaquinaDePilaAbstracta:
    """
    Emulador de Máquina de Pila Virtual basada en evaluación posfija.
    Maneja la pila de evaluación y una memoria de variables simulada.
    """
    def __init__(self):
        self.pila = []
        self.memoria = {}

    def _desapilar(self, operacion):
        """Extrae de la pila con validación de subdesbordamiento (Stack Underflow)."""
        if not self.pila:
            raise RuntimeError(f"Error en tiempo de ejecución: Pila vacía al intentar ejecutar '{operacion}'.")
        return self.pila.pop()

    def ejecutar_instrucciones(self, programa):
        print("=== INICIO DE EJECUCIÓN EN MÁQUINA DE PILA ===")
        for num_linea, instruccion in enumerate(programa, 1):
            partes = instruccion.split()
            if not partes:
                continue
            op = partes[0].upper()

            if op in ("PUSH_NUM", "LOAD_VAR", "STORE_VAR") and len(partes) != 2:
                raise SyntaxError(f"Línea {num_linea}: La instrucción '{op}' requiere exactamente un argumento.")

            if op == "PUSH_NUM":
                val = float(partes[1])
                self.pila.append(val)
                print(f"PUSH_NUM {val:<5} -> Pila: {self.pila}")

            elif op == "LOAD_VAR":
                var_nombre = partes[1]
                if var_nombre not in self.memoria:
                    raise NameError(f"Línea {num_linea}: Acceso ilegal a variable no inicializada '{var_nombre}'.")
                val = self.memoria[var_nombre]
                self.pila.append(val)
                print(f"LOAD_VAR {var_nombre:<5} ({val}) -> Pila: {self.pila}")

            elif op == "STORE_VAR":
                var_nombre = partes[1]
                val = self._desapilar(op)
                self.memoria[var_nombre] = val
                print(f"STORE_VAR {var_nombre:<4} ({val}) -> Pila: {self.pila}")

            elif op in ("ADD", "SUB", "MUL", "DIV"):
                b = self._desapilar(op)  # Operando derecho (cima)
                a = self._desapilar(op)  # Operando izquierdo
                if op == "ADD":
                    res = a + b
                elif op == "SUB":
                    res = a - b
                elif op == "MUL":
                    res = a * b
                elif op == "DIV":
                    if b == 0:
                        raise ZeroDivisionError(f"Línea {num_linea}: División aritmética entre cero.")
                    res = a / b
                self.pila.append(res)
                print(f"{op:<12} ({a} {op} {b}) -> Pila: {self.pila}")

            else:
                raise SyntaxError(f"Línea {num_linea}: Opcode no reconocido '{op}'.")

        print("=== FIN DE EJECUCIÓN CON ÉXITO ===")


if __name__ == "__main__":
    codigo_pila = [
        "PUSH_NUM 5.0",
        "STORE_VAR x",
        "PUSH_NUM 10.0",
        "STORE_VAR y",
        "LOAD_VAR x",
        "LOAD_VAR y",
        "ADD",
        "PUSH_NUM 2.0",
        "MUL",
        "STORE_VAR resultado"
    ]

    vm = MaquinaDePilaAbstracta()
    vm.ejecutar_instrucciones(codigo_pila)
    print("\nEstado final de la memoria principal:", vm.memoria)

```


In [ ]:

class MaquinaDePilaAbstracta:
    """
    Emulador de Máquina de Pila Virtual basada en evaluación posfija.
    Maneja la pila de evaluación y una memoria de variables simulada.
    """
    def __init__(self):
        self.pila = []
        self.memoria = {}

    def _desapilar(self, operacion):
        """Extrae de la pila con validación de subdesbordamiento (Stack Underflow)."""
        if not self.pila:
            raise RuntimeError(f"Error en tiempo de ejecución: Pila vacía al intentar ejecutar '{operacion}'.")
        return self.pila.pop()

    def ejecutar_instrucciones(self, programa):
        print("=== INICIO DE EJECUCIÓN EN MÁQUINA DE PILA ===")
        for num_linea, instruccion in enumerate(programa, 1):
            partes = instruccion.split()
            if not partes:
                continue
            op = partes[0].upper()

            if op in ("PUSH_NUM", "LOAD_VAR", "STORE_VAR") and len(partes) != 2:
                raise SyntaxError(f"Línea {num_linea}: La instrucción '{op}' requiere exactamente un argumento.")

            if op == "PUSH_NUM":
                val = float(partes[1])
                self.pila.append(val)
                print(f"PUSH_NUM {val:<5} -> Pila: {self.pila}")

            elif op == "LOAD_VAR":
                var_nombre = partes[1]
                if var_nombre not in self.memoria:
                    raise NameError(f"Línea {num_linea}: Acceso ilegal a variable no inicializada '{var_nombre}'.")
                val = self.memoria[var_nombre]
                self.pila.append(val)
                print(f"LOAD_VAR {var_nombre:<5} ({val}) -> Pila: {self.pila}")

            elif op == "STORE_VAR":
                var_nombre = partes[1]
                val = self._desapilar(op)
                self.memoria[var_nombre] = val
                print(f"STORE_VAR {var_nombre:<4} ({val}) -> Pila: {self.pila}")

            elif op in ("ADD", "SUB", "MUL", "DIV"):
                b = self._desapilar(op)  # Operando derecho (cima)
                a = self._desapilar(op)  # Operando izquierdo
                if op == "ADD":
                    res = a + b
                elif op == "SUB":
                    res = a - b
                elif op == "MUL":
                    res = a * b
                elif op == "DIV":
                    if b == 0:
                        raise ZeroDivisionError(f"Línea {num_linea}: División aritmética entre cero.")
                    res = a / b
                self.pila.append(res)
                print(f"{op:<12} ({a} {op} {b}) -> Pila: {self.pila}")

            else:
                raise SyntaxError(f"Línea {num_linea}: Opcode no reconocido '{op}'.")

        print("=== FIN DE EJECUCIÓN CON ÉXITO ===")


if __name__ == "__main__":
    codigo_pila = [
        "PUSH_NUM 5.0",
        "STORE_VAR x",
        "PUSH_NUM 10.0",
        "STORE_VAR y",
        "LOAD_VAR x",
        "LOAD_VAR y",
        "ADD",
        "PUSH_NUM 2.0",
        "MUL",
        "STORE_VAR resultado"
    ]

    vm = MaquinaDePilaAbstracta()
    vm.ejecutar_instrucciones(codigo_pila)
    print("\nEstado final de la memoria principal:", vm.memoria)


=== INICIO DE EJECUCIÓN EN MÁQUINA DE PILA ===
PUSH_NUM 5.0   -> Pila: [5.0]
STORE_VAR x    (5.0) -> Pila: []
PUSH_NUM 10.0  -> Pila: [10.0]
STORE_VAR y    (10.0) -> Pila: []
LOAD_VAR x     (5.0) -> Pila: [5.0]
LOAD_VAR y     (10.0) -> Pila: [5.0, 10.0]
ADD          (5.0 ADD 10.0) -> Pila: [15.0]
PUSH_NUM 2.0   -> Pila: [15.0, 2.0]
MUL          (15.0 MUL 2.0) -> Pila: [30.0]
STORE_VAR resultado (30.0) -> Pila: []
=== FIN DE EJECUCIÓN CON ÉXITO ===

Estado final de la memoria principal: {'x': 5.0, 'y': 10.0, 'resultado': 30.0}


## 4. Sección Práctica 2: Tabla de Símbolos Jerárquica (Gestión de Ámbitos / Scopes) y Verificador de Errores Semánticos

Para manejar estructuras de control o funciones en los lenguajes originales de los equipos, la Tabla de Símbolos debe gestionar niveles de anidamiento (Scopes).

### [CÓDIGO 2.1] Implementación de la Tabla de Símbolos Jerárquica y verificación de errores semánticos
```python

class ErrorSemantico(Exception):
    """Excepción base para errores semánticos."""
    pass


class Simbolo:
    """Representa una entrada formal en la Tabla de Símbolos."""
    def __init__(self, nombre, tipo_dato, categoria="variable", offset=0):
        self.nombre = nombre
        self.tipo_dato = tipo_dato
        self.categoria = categoria
        self.offset = offset

    def __repr__(self):
        return f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', cat='{self.categoria}', offset={self.offset})"


class EntornoAmbito:
    """
    Representa un bloque o ámbito local específico dentro del programa.
    Mantiene un diccionario local de símbolos, enlace a su ámbito padre y offset base.
    """
    def __init__(self, id_bloque, padre=None, nivel_profundidad=0, offset_base=0):
        self.id_bloque = id_bloque
        self.padre = padre
        self.nivel_profundidad = nivel_profundidad
        self.offset_base = offset_base  # Posición inicial de memoria al abrir el bloque
        self.tabla_local = {}

    def insertar_local(self, simbolo):
        if simbolo.nombre in self.tabla_local:
            raise ErrorSemantico(
                f"Error Semántico [Bloque {self.id_bloque}, Nivel {self.nivel_profundidad}]: "
                f"Redeclaración ilegal de la variable '{simbolo.nombre}' en el mismo ámbito local."
            )
        self.tabla_local[simbolo.nombre] = simbolo

    def buscar_local(self, nombre):
        return self.tabla_local.get(nombre, None)


# Tabla de tamaños en bytes según tipo de dato
TABLA_TAMANOS_BYTES = {
    "int": 4,
    "float": 4,
    "double": 8,
    "char": 1,
    "bool": 1,
    "string": 8
}


class TablaSimbolosJerarquica:
    """
    Administrador central de la Tabla de Símbolos Jerárquica con soporte
    para bloques anidados (Scopes) y cálculo de direcciones relativas (Offsets).
    """
    def __init__(self):
        self.contador_bloques = 0
        self.offset_acumulado = 0
        # Ámbito global inicial (Bloque 0, Nivel 0)
        self.ambito_actual = EntornoAmbito(id_bloque=0, padre=None, nivel_profundidad=0, offset_base=0)

    def abrir_bloque(self, nombre_contexto=""):
        self.contador_bloques += 1
        nuevo_nivel = self.ambito_actual.nivel_profundidad + 1
        nuevo_entorno = EntornoAmbito(
            id_bloque=self.contador_bloques,
            padre=self.ambito_actual,
            nivel_profundidad=nuevo_nivel,
            offset_base=self.offset_acumulado
        )
        self.ambito_actual = nuevo_entorno
        tag = f" ({nombre_contexto})" if nombre_contexto else ""
        print(f"\n[+] ABRIR BLOQUE {self.ambito_actual.id_bloque}{tag} | Nivel de profundidad: {nuevo_nivel}")

    def cerrar_bloque(self):
        if self.ambito_actual.padre is None:
            raise ErrorSemantico("Error: Intentando cerrar el ámbito global del programa.")

        bytes_liberados = self.offset_acumulado - self.ambito_actual.offset_base
        print(f"\n[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a Nivel {self.ambito_actual.padre.nivel_profundidad}"
              f" | Memoria liberada: {bytes_liberados} bytes")
        # Liberación de memoria tipo pila: se reutiliza el espacio de variables locales
        self.offset_acumulado = self.ambito_actual.offset_base
        self.ambito_actual = self.ambito_actual.padre

    def insertar(self, nombre, tipo_dato, categoria="variable", tamano_bytes=None):
        if tamano_bytes is None:
            tamano_bytes = TABLA_TAMANOS_BYTES.get(tipo_dato, 4)
        simbolo = Simbolo(nombre, tipo_dato, categoria, self.offset_acumulado)
        self.ambito_actual.insertar_local(simbolo)
        self.offset_acumulado += tamano_bytes
        print(f"   ✓ SIMBOLO REGISTRADO: '{nombre}' -> {simbolo} (Ámbito {self.ambito_actual.id_bloque})")
        return simbolo

    def buscar(self, nombre):
        actual = self.ambito_actual
        while actual is not None:
            simbolo = actual.buscar_local(nombre)
            if simbolo is not None:
                print(f"   🔍 BUSQUEDA EXITOSA: '{nombre}' encontrado en Bloque {actual.id_bloque} (Nivel {actual.nivel_profundidad}) -> {simbolo}")
                return simbolo
            actual = actual.padre

        raise ErrorSemantico(
            f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: "
            f"La variable '{nombre}' no ha sido declarada en el ámbito activo ni en sus ámbitos contenedores."
        )

    def mostrar_cadena_ambitos(self):
        print("\n" + "=" * 70)
        print("ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:")
        print("=" * 70)
        actual = self.ambito_actual
        while actual is not None:
            simbolos = list(actual.tabla_local.keys())
            print(f" ► [Bloque {actual.id_bloque} | Nivel {actual.nivel_profundidad}] Símbolos locales: {{{', '.join(simbolos)}}}")
            actual = actual.padre
        print("=" * 70 + "\n")


if __name__ == "__main__":
    print("=" * 70)
    print("DEMOSTRACIÓN: TABLA DE SÍMBOLOS CON GESTIÓN DE ÁMBITOS Y ANIDAMIENTO")
    print("=" * 70)

    ts = TablaSimbolosJerarquica()

    # 1. Variables globales
    print("\n--- 1. Declaraciones en Ámbito Global ---")
    ts.insertar("global_x", "int", categoria="variable")      # offset 0, tamaño 4
    ts.insertar("PI", "float", categoria="constante")         # offset 4, tamaño 4
    ts.mostrar_cadena_ambitos()

    # 2. Función main (Bloque 1)
    ts.abrir_bloque("Funcion main")
    ts.insertar("a", "int")                                   # offset 8, tamaño 4
    ts.insertar("b", "float")                                 # offset 12, tamaño 4
    ts.buscar("global_x")
    ts.mostrar_cadena_ambitos()

    # 3. Bloque anidado (Bloque 2)
    ts.abrir_bloque("Bloque interno IF")
    ts.insertar("a", "string")                                # Sombreado de 'a': string = 8 bytes -> offset 16
    ts.insertar("temp", "int")                                # offset 16 + 8 = 24
    ts.mostrar_cadena_ambitos()

    # Búsquedas jerárquicas
    ts.buscar("a")         # Local en Bloque 2
    ts.buscar("b")         # Encontrado en Bloque 1
    ts.buscar("global_x")  # Encontrado en Bloque 0

    # 4. Pruebas de errores semánticos
    print("\n--- 4. Demostración de Captura de Errores Semánticos ---")

    print("\n   [Prueba A] Intentando redeclarar 'temp' en el mismo bloque local...")
    try:
        ts.insertar("temp", "int")
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # Cierre de bloque 2 (libera 12 bytes: 8 de 'a' string + 4 de 'temp' int)
    ts.cerrar_bloque()
    ts.buscar("a")  # Ahora 'a' vuelve a ser la de Bloque 1 (int, offset 8)

    print("\n   [Prueba B] Intentando buscar 'temp' fuera de su ámbito...")
    try:
        ts.buscar("temp")
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # Cierre de main (libera 8 bytes)
    ts.cerrar_bloque()
    ts.mostrar_cadena_ambitos()

    # 5. Demostración de reutilización de memoria (Stack Frame reuse)
    print("\n--- 5. Reutilización de Espacio de Memoria Liberado ---")
    ts.abrir_bloque("Funcion auxiliar")
    ts.insertar("z", "int")  # Reutiliza offset 8
    ts.cerrar_bloque()

    print("\n   [Prueba C] Intentando cerrar el ámbito global...")
    try:
        ts.cerrar_bloque()
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")

```


In [ ]:

class ErrorSemantico(Exception):
    """Excepción base para errores semánticos."""
    pass


class Simbolo:
    """Representa una entrada formal en la Tabla de Símbolos."""
    def __init__(self, nombre, tipo_dato, categoria="variable", offset=0):
        self.nombre = nombre
        self.tipo_dato = tipo_dato
        self.categoria = categoria
        self.offset = offset

    def __repr__(self):
        return f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', cat='{self.categoria}', offset={self.offset})"


class EntornoAmbito:
    """
    Representa un bloque o ámbito local específico dentro del programa.
    Mantiene un diccionario local de símbolos, enlace a su ámbito padre y offset base.
    """
    def __init__(self, id_bloque, padre=None, nivel_profundidad=0, offset_base=0):
        self.id_bloque = id_bloque
        self.padre = padre
        self.nivel_profundidad = nivel_profundidad
        self.offset_base = offset_base  # Posición inicial de memoria al abrir el bloque
        self.tabla_local = {}

    def insertar_local(self, simbolo):
        if simbolo.nombre in self.tabla_local:
            raise ErrorSemantico(
                f"Error Semántico [Bloque {self.id_bloque}, Nivel {self.nivel_profundidad}]: "
                f"Redeclaración ilegal de la variable '{simbolo.nombre}' en el mismo ámbito local."
            )
        self.tabla_local[simbolo.nombre] = simbolo

    def buscar_local(self, nombre):
        return self.tabla_local.get(nombre, None)


# Tabla de tamaños en bytes según tipo de dato
TABLA_TAMANOS_BYTES = {
    "int": 4,
    "float": 4,
    "double": 8,
    "char": 1,
    "bool": 1,
    "string": 8
}


class TablaSimbolosJerarquica:
    """
    Administrador central de la Tabla de Símbolos Jerárquica con soporte
    para bloques anidados (Scopes) y cálculo de direcciones relativas (Offsets).
    """
    def __init__(self):
        self.contador_bloques = 0
        self.offset_acumulado = 0
        # Ámbito global inicial (Bloque 0, Nivel 0)
        self.ambito_actual = EntornoAmbito(id_bloque=0, padre=None, nivel_profundidad=0, offset_base=0)

    def abrir_bloque(self, nombre_contexto=""):
        self.contador_bloques += 1
        nuevo_nivel = self.ambito_actual.nivel_profundidad + 1
        nuevo_entorno = EntornoAmbito(
            id_bloque=self.contador_bloques,
            padre=self.ambito_actual,
            nivel_profundidad=nuevo_nivel,
            offset_base=self.offset_acumulado
        )
        self.ambito_actual = nuevo_entorno
        tag = f" ({nombre_contexto})" if nombre_contexto else ""
        print(f"\n[+] ABRIR BLOQUE {self.ambito_actual.id_bloque}{tag} | Nivel de profundidad: {nuevo_nivel}")

    def cerrar_bloque(self):
        if self.ambito_actual.padre is None:
            raise ErrorSemantico("Error: Intentando cerrar el ámbito global del programa.")

        bytes_liberados = self.offset_acumulado - self.ambito_actual.offset_base
        print(f"\n[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a Nivel {self.ambito_actual.padre.nivel_profundidad}"
              f" | Memoria liberada: {bytes_liberados} bytes")
        # Liberación de memoria tipo pila: se reutiliza el espacio de variables locales
        self.offset_acumulado = self.ambito_actual.offset_base
        self.ambito_actual = self.ambito_actual.padre

    def insertar(self, nombre, tipo_dato, categoria="variable", tamano_bytes=None):
        if tamano_bytes is None:
            tamano_bytes = TABLA_TAMANOS_BYTES.get(tipo_dato, 4)
        simbolo = Simbolo(nombre, tipo_dato, categoria, self.offset_acumulado)
        self.ambito_actual.insertar_local(simbolo)
        self.offset_acumulado += tamano_bytes
        print(f"   ✓ SIMBOLO REGISTRADO: '{nombre}' -> {simbolo} (Ámbito {self.ambito_actual.id_bloque})")
        return simbolo

    def buscar(self, nombre):
        actual = self.ambito_actual
        while actual is not None:
            simbolo = actual.buscar_local(nombre)
            if simbolo is not None:
                print(f"   🔍 BUSQUEDA EXITOSA: '{nombre}' encontrado en Bloque {actual.id_bloque} (Nivel {actual.nivel_profundidad}) -> {simbolo}")
                return simbolo
            actual = actual.padre

        raise ErrorSemantico(
            f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: "
            f"La variable '{nombre}' no ha sido declarada en el ámbito activo ni en sus ámbitos contenedores."
        )

    def mostrar_cadena_ambitos(self):
        print("\n" + "=" * 70)
        print("ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:")
        print("=" * 70)
        actual = self.ambito_actual
        while actual is not None:
            simbolos = list(actual.tabla_local.keys())
            print(f" ► [Bloque {actual.id_bloque} | Nivel {actual.nivel_profundidad}] Símbolos locales: {{{', '.join(simbolos)}}}")
            actual = actual.padre
        print("=" * 70 + "\n")


if __name__ == "__main__":
    print("=" * 70)
    print("DEMOSTRACIÓN: TABLA DE SÍMBOLOS CON GESTIÓN DE ÁMBITOS Y ANIDAMIENTO")
    print("=" * 70)

    ts = TablaSimbolosJerarquica()

    # 1. Variables globales
    print("\n--- 1. Declaraciones en Ámbito Global ---")
    ts.insertar("global_x", "int", categoria="variable")      # offset 0, tamaño 4
    ts.insertar("PI", "float", categoria="constante")         # offset 4, tamaño 4
    ts.mostrar_cadena_ambitos()

    # 2. Función main (Bloque 1)
    ts.abrir_bloque("Funcion main")
    ts.insertar("a", "int")                                   # offset 8, tamaño 4
    ts.insertar("b", "float")                                 # offset 12, tamaño 4
    ts.buscar("global_x")
    ts.mostrar_cadena_ambitos()

    # 3. Bloque anidado (Bloque 2)
    ts.abrir_bloque("Bloque interno IF")
    ts.insertar("a", "string")                                # Sombreado de 'a': string = 8 bytes -> offset 16
    ts.insertar("temp", "int")                                # offset 16 + 8 = 24
    ts.mostrar_cadena_ambitos()

    # Búsquedas jerárquicas
    ts.buscar("a")         # Local en Bloque 2
    ts.buscar("b")         # Encontrado en Bloque 1
    ts.buscar("global_x")  # Encontrado en Bloque 0

    # 4. Pruebas de errores semánticos
    print("\n--- 4. Demostración de Captura de Errores Semánticos ---")

    print("\n   [Prueba A] Intentando redeclarar 'temp' en el mismo bloque local...")
    try:
        ts.insertar("temp", "int")
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # Cierre de bloque 2 (libera 12 bytes: 8 de 'a' string + 4 de 'temp' int)
    ts.cerrar_bloque()
    ts.buscar("a")  # Ahora 'a' vuelve a ser la de Bloque 1 (int, offset 8)

    print("\n   [Prueba B] Intentando buscar 'temp' fuera de su ámbito...")
    try:
        ts.buscar("temp")
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")

    # Cierre de main (libera 8 bytes)
    ts.cerrar_bloque()
    ts.mostrar_cadena_ambitos()

    # 5. Demostración de reutilización de memoria (Stack Frame reuse)
    print("\n--- 5. Reutilización de Espacio de Memoria Liberado ---")
    ts.abrir_bloque("Funcion auxiliar")
    ts.insertar("z", "int")  # Reutiliza offset 8
    ts.cerrar_bloque()

    print("\n   [Prueba C] Intentando cerrar el ámbito global...")
    try:
        ts.cerrar_bloque()
    except ErrorSemantico as e:
        print(f"     [CAPTURADO EXITO] {e}")


DEMOSTRACIÓN: TABLA DE SÍMBOLOS CON GESTIÓN DE ÁMBITOS Y ANIDAMIENTO

--- 1. Declaraciones en Ámbito Global ---
   ✓ SIMBOLO REGISTRADO: 'global_x' -> Simbolo('global_x', tipo='int', cat='variable', offset=0) (Ámbito 0)
   ✓ SIMBOLO REGISTRADO: 'PI' -> Simbolo('PI', tipo='float', cat='constante', offset=4) (Ámbito 0)

ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:
 ► [Bloque 0 | Nivel 0] Símbolos locales: {global_x, PI}


[+] ABRIR BLOQUE 1 (Funcion main) | Nivel de profundidad: 1
   ✓ SIMBOLO REGISTRADO: 'a' -> Simbolo('a', tipo='int', cat='variable', offset=8) (Ámbito 1)
   ✓ SIMBOLO REGISTRADO: 'b' -> Simbolo('b', tipo='float', cat='variable', offset=12) (Ámbito 1)
   🔍 BUSQUEDA EXITOSA: 'global_x' encontrado en Bloque 0 (Nivel 0) -> Simbolo('global_x', tipo='int', cat='variable', offset=0)

ESTADO ACTUAL DE LA CADENA DE ÁMBITOS VISIBLES:
 ► [Bloque 1 | Nivel 1] Símbolos locales: {a, b}
 ► [Bloque 0 | Nivel 0] Símbolos locales: {global_x, PI}


[+] ABRIR BLOQUE 2 (Bloque interno IF

## 5. Sección Práctica 3:  Integración del módulo de la tabla de símbolos con una clase de Árbol de Sintaxis Abstracta (AST)

En esta sección implementaremos el código para integrar la Tabla de Símbolos con una clase del Árbol de Sintaxis Abstracta (AST) para realizar una validación semántica completa de asignaciones y expresiones.

### [CÓDIGO 5.1] Integración de la Tabla de Símbolos con AST.
```python

# ==============================================================================
# PROGRAMACIÓN DE SISTEMAS DE BASE II
# PRÁCTICA INTEGRADORA: ÁRBOLES DE SINTAXIS ABSTRACTA (AST) + TABLA DE SÍMBOLOS
# Modalidad: POO Nativa Pura en Python (Sin librerías externas)
# ==============================================================================

class ErrorSemantico(Exception):
    """Excepción base para errores semánticos."""
    pass


class ErrorRedeclaracion(ErrorSemantico):
    """Error al intentar redeclarar una variable en el mismo ámbito."""
    pass


class ErrorVariableNoDeclarada(ErrorSemantico):
    """Error al intentar usar una variable no existente en el contexto."""
    pass


class ErrorTipoIncompatible(ErrorSemantico):
    """Error al operar o asignar tipos de datos incompatibles."""
    pass


# ------------------------------------------------------------------------------
# 1. MÓDULO DE LA TABLA DE SÍMBOLOS Y ÁMBITOS
# ------------------------------------------------------------------------------

class Simbolo:
    def __init__(self, nombre, tipo_dato, valor=None, categoria="variable", offset=0):
        self.nombre = nombre
        self.tipo_dato = tipo_dato  # 'int', 'float', 'string'
        self.valor = valor
        self.categoria = categoria
        self.offset = offset

    def __repr__(self):
        return f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', val={self.valor}, offset={self.offset})"


class EntornoAmbito:
    def __init__(self, id_bloque, padre=None, nivel=0, offset_base=0):
        self.id_bloque = id_bloque
        self.padre = padre
        self.nivel = nivel
        self.offset_base = offset_base
        self.tabla_local = {}

    def insertar_local(self, simbolo):
        if simbolo.nombre in self.tabla_local:
            raise ErrorRedeclaracion(
                f"Error Semántico [Bloque {self.id_bloque}]: Redeclaración ilegal de la variable '{simbolo.nombre}' en el mismo ámbito."
            )
        self.tabla_local[simbolo.nombre] = simbolo

    def buscar_local(self, nombre):
        return self.tabla_local.get(nombre, None)


class TablaSimbolosJerarquica:
    def __init__(self):
        self.contador_bloques = 0
        self.offset_acumulado = 0
        self.ambito_actual = EntornoAmbito(id_bloque=0, padre=None, nivel=0, offset_base=0)

    def abrir_bloque(self, etiqueta=""):
        self.contador_bloques += 1
        nuevo_entorno = EntornoAmbito(
            id_bloque=self.contador_bloques,
            padre=self.ambito_actual,
            nivel=self.ambito_actual.nivel + 1,
            offset_base=self.offset_acumulado
        )
        self.ambito_actual = nuevo_entorno
        tag = f" ({etiqueta})" if etiqueta else ""
        print(f"\n[+] ABRIR BLOQUE {self.ambito_actual.id_bloque}{tag} | Nivel {self.ambito_actual.nivel}")

    def cerrar_bloque(self):
        if self.ambito_actual.padre is None:
            raise ErrorSemantico("Error: No se puede cerrar el ámbito global.")
        print(f"[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a Nivel {self.ambito_actual.padre.nivel}")
        self.offset_acumulado = self.ambito_actual.offset_base
        self.ambito_actual = self.ambito_actual.padre

    def declarar(self, nombre, tipo_dato, valor_inicial=None, tamano=4):
        simbolo = Simbolo(nombre, tipo_dato, valor_inicial, offset=self.offset_acumulado)
        self.ambito_actual.insertar_local(simbolo)
        self.offset_acumulado += tamano
        print(f"   ✓ DECLARACIÓN: Variable '{nombre}' ({tipo_dato}) registrada en Bloque {self.ambito_actual.id_bloque}")
        return simbolo

    def buscar(self, nombre):
        actual = self.ambito_actual
        while actual is not None:
            simbolo = actual.buscar_local(nombre)
            if simbolo is not None:
                return simbolo
            actual = actual.padre
        raise ErrorVariableNoDeclarada(
            f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: La variable '{nombre}' no ha sido declarada en el ámbito activo ni en sus contenedores."
        )


# ------------------------------------------------------------------------------
# 2. COMPROBACIÓN Y COERCIÓN SEMÁNTICA DE TIPOS
# ------------------------------------------------------------------------------

TIPOS_NUMERICOS = ("int", "float")


def validar_y_coercionar_asignacion(tipo_destino, valor, tipo_origen, nombre_var):
    """
    Aplica las reglas de tipado estricto y coerciones implícitas:
    - Tipos iguales: Asignación permitida sin cambios.
    - float -> int: Truncamiento con advertencia (coerción con pérdida).
    - int -> float: Promoción implícita a float.
    - Incompatibilidad (string con numérico): Lanza ErrorTipoIncompatible.
    """
    if tipo_destino == tipo_origen:
        return valor
    if tipo_destino in TIPOS_NUMERICOS and tipo_origen in TIPOS_NUMERICOS:
        if tipo_destino == "int":
            print(f"   [COERCIÓN ADVERTENCIA]: Truncando 'float' a 'int' para asignación en '{nombre_var}'")
            return int(valor)
        return float(valor)
    raise ErrorTipoIncompatible(
        f"Error Semántico de Tipos: No se puede asignar '{tipo_origen}' a la variable '{nombre_var}' de tipo '{tipo_destino}'."
    )


# ------------------------------------------------------------------------------
# 3. NODOS DEL ÁRBOL DE SINTAXIS ABSTRACTA (AST)
# ------------------------------------------------------------------------------

class NodoAST:
    def evaluar(self, ts: TablaSimbolosJerarquica):
        raise NotImplementedError


class Numero(NodoAST):
    def __init__(self, valor):
        self.valor = valor
        self.tipo_dato = "int" if isinstance(valor, int) else "float"

    def evaluar(self, ts):
        return self.valor, self.tipo_dato


class Variable(NodoAST):
    def __init__(self, nombre):
        self.nombre = nombre

    def evaluar(self, ts):
        simbolo = ts.buscar(self.nombre)
        if simbolo.valor is None:
            raise ErrorSemantico(f"Error Semántico: La variable '{self.nombre}' no ha sido inicializada.")
        return simbolo.valor, simbolo.tipo_dato


class Operador(NodoAST):
    def __init__(self, operador, izq, der):
        self.operador = operador
        self.izq = izq
        self.der = der

    def evaluar(self, ts):
        val_izq, tipo_izq = self.izq.evaluar(ts)
        val_der, tipo_der = self.der.evaluar(ts)

        if tipo_izq == "string" or tipo_der == "string":
            raise ErrorTipoIncompatible(
                f"Error Semántico de Tipos: Operación '{self.operador}' no permitida entre '{tipo_izq}' y '{tipo_der}'."
            )

        tipo_resultante = "float" if ("float" in (tipo_izq, tipo_der)) else "int"

        if self.operador == "+":
            res = val_izq + val_der
        elif self.operador == "-":
            res = val_izq - val_der
        elif self.operador == "*":
            res = val_izq * val_der
        elif self.operador == "/":
            if val_der == 0:
                raise ZeroDivisionError("Error Semántico/Aritmético: División entre cero.")
            res = val_izq / val_der
            tipo_resultante = "float"
        else:
            raise ErrorSemantico(f"Error Semántico: Operador '{self.operador}' no soportado.")

        return res, tipo_resultante


class Declaracion(NodoAST):
    def __init__(self, nombre, tipo_declarado, expresion_inicial=None):
        self.nombre = nombre
        self.tipo_declarado = tipo_declarado
        self.expresion_inicial = expresion_inicial

    def evaluar(self, ts):
        val_expr = None
        if self.expresion_inicial is not None:
            # Evaluar expresión antes de registrar símbolo para resolver shadowing correctamente
            val_expr, tipo_expr = self.expresion_inicial.evaluar(ts)
            val_expr = validar_y_coercionar_asignacion(self.tipo_declarado, val_expr, tipo_expr, self.nombre)
        ts.declarar(self.nombre, self.tipo_declarado, val_expr)
        return val_expr, self.tipo_declarado


class Asignacion(NodoAST):
    def __init__(self, nombre_var, expresion):
        self.nombre_var = nombre_var
        self.expresion = expresion

    def evaluar(self, ts):
        simbolo = ts.buscar(self.nombre_var)
        val_expr, tipo_expr = self.expresion.evaluar(ts)
        val_expr = validar_y_coercionar_asignacion(simbolo.tipo_dato, val_expr, tipo_expr, self.nombre_var)
        simbolo.valor = val_expr
        print(f"   ✓ ASIGNACIÓN EXITOSA: {self.nombre_var} = {val_expr} (Tipo: {simbolo.tipo_dato})")
        return val_expr, simbolo.tipo_dato


class BloqueAST(NodoAST):
    def __init__(self, instrucciones, etiqueta=""):
        self.instrucciones = instrucciones
        self.etiqueta = etiqueta

    def evaluar(self, ts):
        ts.abrir_bloque(self.etiqueta)
        ultimo_res = None
        try:
            for instr in self.instrucciones:
                ultimo_res = instr.evaluar(ts)
        finally:
            # Garantiza la restauración del ámbito incluso ante excepciones semánticas
            ts.cerrar_bloque()
        return ultimo_res


# ------------------------------------------------------------------------------
# 4. DEMOSTRACIÓN DE EJECUCIÓN Y BATERÍA DE PRUEBAS SEMÁNTICAS
# ------------------------------------------------------------------------------

if __name__ == "__main__":
    ts = TablaSimbolosJerarquica()

    # Variables globales
    ts.declarar("global_a", "int", 10)
    ts.declarar("global_b", "float", 20.5)

    # Programa AST con anidamiento
    programa = BloqueAST(
        [
            Declaracion("x", "int", Operador("*", Variable("global_a"), Numero(2))),
            Asignacion("x", Operador("+", Variable("x"), Numero(5))),
            BloqueAST(
                [Declaracion("x", "float", Numero(3.14))],
                etiqueta="Bloque Anidado"
            ),
        ],
        etiqueta="Programa Principal"
    )

    print("--- Ejecutando Evaluación del AST e Integración con Tabla de Símbolos ---")
    programa.evaluar(ts)

    # Batería exhaustiva de pruebas semánticas
    print("\n--- Pruebas de Captura de Errores Semánticos ---")
    ts.declarar("texto", "string", "hola")
    ts.declarar("sin_valor", "int")

    casos_de_prueba = [
        ("A. Operación int + string", Operador("+", Variable("global_a"), Variable("texto"))),
        ("B. Inicializar float con string", Declaracion("f", "float", Variable("texto"))),
        ("C. Inicializar string con int", Declaracion("s", "string", Numero(5))),
        ("D. Variable no declarada", Variable("no_existe")),
        ("E. Variable sin inicializar", Variable("sin_valor")),
        ("F. Redeclaración en el mismo ámbito", Declaracion("global_a", "int", Numero(1))),
        ("G. Operador no soportado", Operador("%", Numero(5), Numero(2))),
        ("H. División entre cero", Operador("/", Numero(1), Numero(0))),
        ("I. Error dentro de bloque anidado (comprobando cierre seguro)",
         BloqueAST([Variable("no_existe")], etiqueta="Bloque con fallo")),
    ]

    for etiqueta, nodo in casos_de_prueba:
        print(f"\n   [Prueba {etiqueta}]")
        try:
            nodo.evaluar(ts)
            print("     ✗ Error no detectado")
        except (ErrorSemantico, ZeroDivisionError) as err:
            print(f"     [CAPTURADO ÉXITO]: {err}")

    print(f"\n   Ámbito activo final: Bloque {ts.ambito_actual.id_bloque} (Nivel {ts.ambito_actual.nivel})")

    # Demostración de coerciones numéricas
    print("\n--- Pruebas de Coerciones Numéricas Válidas ---")
    Declaracion("entero", "int", Numero(2.9)).evaluar(ts)
    Declaracion("real", "float", Variable("global_a")).evaluar(ts)
    print(f"   entero = {ts.buscar('entero').valor}, real = {ts.buscar('real').valor}")

```


In [ ]:

# ==============================================================================
# PROGRAMACIÓN DE SISTEMAS DE BASE II
# PRÁCTICA INTEGRADORA: ÁRBOLES DE SINTAXIS ABSTRACTA (AST) + TABLA DE SÍMBOLOS
# Modalidad: POO Nativa Pura en Python (Sin librerías externas)
# ==============================================================================

class ErrorSemantico(Exception):
    """Excepción base para errores semánticos."""
    pass


class ErrorRedeclaracion(ErrorSemantico):
    """Error al intentar redeclarar una variable en el mismo ámbito."""
    pass


class ErrorVariableNoDeclarada(ErrorSemantico):
    """Error al intentar usar una variable no existente en el contexto."""
    pass


class ErrorTipoIncompatible(ErrorSemantico):
    """Error al operar o asignar tipos de datos incompatibles."""
    pass


# ------------------------------------------------------------------------------
# 1. MÓDULO DE LA TABLA DE SÍMBOLOS Y ÁMBITOS
# ------------------------------------------------------------------------------

class Simbolo:
    def __init__(self, nombre, tipo_dato, valor=None, categoria="variable", offset=0):
        self.nombre = nombre
        self.tipo_dato = tipo_dato  # 'int', 'float', 'string'
        self.valor = valor
        self.categoria = categoria
        self.offset = offset

    def __repr__(self):
        return f"Simbolo('{self.nombre}', tipo='{self.tipo_dato}', val={self.valor}, offset={self.offset})"


class EntornoAmbito:
    def __init__(self, id_bloque, padre=None, nivel=0, offset_base=0):
        self.id_bloque = id_bloque
        self.padre = padre
        self.nivel = nivel
        self.offset_base = offset_base
        self.tabla_local = {}

    def insertar_local(self, simbolo):
        if simbolo.nombre in self.tabla_local:
            raise ErrorRedeclaracion(
                f"Error Semántico [Bloque {self.id_bloque}]: Redeclaración ilegal de la variable '{simbolo.nombre}' en el mismo ámbito."
            )
        self.tabla_local[simbolo.nombre] = simbolo

    def buscar_local(self, nombre):
        return self.tabla_local.get(nombre, None)


class TablaSimbolosJerarquica:
    def __init__(self):
        self.contador_bloques = 0
        self.offset_acumulado = 0
        self.ambito_actual = EntornoAmbito(id_bloque=0, padre=None, nivel=0, offset_base=0)

    def abrir_bloque(self, etiqueta=""):
        self.contador_bloques += 1
        nuevo_entorno = EntornoAmbito(
            id_bloque=self.contador_bloques,
            padre=self.ambito_actual,
            nivel=self.ambito_actual.nivel + 1,
            offset_base=self.offset_acumulado
        )
        self.ambito_actual = nuevo_entorno
        tag = f" ({etiqueta})" if etiqueta else ""
        print(f"\n[+] ABRIR BLOQUE {self.ambito_actual.id_bloque}{tag} | Nivel {self.ambito_actual.nivel}")

    def cerrar_bloque(self):
        if self.ambito_actual.padre is None:
            raise ErrorSemantico("Error: No se puede cerrar el ámbito global.")
        print(f"[-] CERRAR BLOQUE {self.ambito_actual.id_bloque} | Retornando a Nivel {self.ambito_actual.padre.nivel}")
        self.offset_acumulado = self.ambito_actual.offset_base
        self.ambito_actual = self.ambito_actual.padre

    def declarar(self, nombre, tipo_dato, valor_inicial=None, tamano=4):
        simbolo = Simbolo(nombre, tipo_dato, valor_inicial, offset=self.offset_acumulado)
        self.ambito_actual.insertar_local(simbolo)
        self.offset_acumulado += tamano
        print(f"   ✓ DECLARACIÓN: Variable '{nombre}' ({tipo_dato}) registrada en Bloque {self.ambito_actual.id_bloque}")
        return simbolo

    def buscar(self, nombre):
        actual = self.ambito_actual
        while actual is not None:
            simbolo = actual.buscar_local(nombre)
            if simbolo is not None:
                return simbolo
            actual = actual.padre
        raise ErrorVariableNoDeclarada(
            f"Error Semántico [Bloque {self.ambito_actual.id_bloque}]: La variable '{nombre}' no ha sido declarada en el ámbito activo ni en sus contenedores."
        )


# ------------------------------------------------------------------------------
# 2. COMPROBACIÓN Y COERCIÓN SEMÁNTICA DE TIPOS
# ------------------------------------------------------------------------------

TIPOS_NUMERICOS = ("int", "float")


def validar_y_coercionar_asignacion(tipo_destino, valor, tipo_origen, nombre_var):
    """
    Aplica las reglas de tipado estricto y coerciones implícitas:
    - Tipos iguales: Asignación permitida sin cambios.
    - float -> int: Truncamiento con advertencia (coerción con pérdida).
    - int -> float: Promoción implícita a float.
    - Incompatibilidad (string con numérico): Lanza ErrorTipoIncompatible.
    """
    if tipo_destino == tipo_origen:
        return valor
    if tipo_destino in TIPOS_NUMERICOS and tipo_origen in TIPOS_NUMERICOS:
        if tipo_destino == "int":
            print(f"   [COERCIÓN ADVERTENCIA]: Truncando 'float' a 'int' para asignación en '{nombre_var}'")
            return int(valor)
        return float(valor)
    raise ErrorTipoIncompatible(
        f"Error Semántico de Tipos: No se puede asignar '{tipo_origen}' a la variable '{nombre_var}' de tipo '{tipo_destino}'."
    )


# ------------------------------------------------------------------------------
# 3. NODOS DEL ÁRBOL DE SINTAXIS ABSTRACTA (AST)
# ------------------------------------------------------------------------------

class NodoAST:
    def evaluar(self, ts: TablaSimbolosJerarquica):
        raise NotImplementedError


class Numero(NodoAST):
    def __init__(self, valor):
        self.valor = valor
        self.tipo_dato = "int" if isinstance(valor, int) else "float"

    def evaluar(self, ts):
        return self.valor, self.tipo_dato


class Variable(NodoAST):
    def __init__(self, nombre):
        self.nombre = nombre

    def evaluar(self, ts):
        simbolo = ts.buscar(self.nombre)
        if simbolo.valor is None:
            raise ErrorSemantico(f"Error Semántico: La variable '{self.nombre}' no ha sido inicializada.")
        return simbolo.valor, simbolo.tipo_dato


class Operador(NodoAST):
    def __init__(self, operador, izq, der):
        self.operador = operador
        self.izq = izq
        self.der = der

    def evaluar(self, ts):
        val_izq, tipo_izq = self.izq.evaluar(ts)
        val_der, tipo_der = self.der.evaluar(ts)

        if tipo_izq == "string" or tipo_der == "string":
            raise ErrorTipoIncompatible(
                f"Error Semántico de Tipos: Operación '{self.operador}' no permitida entre '{tipo_izq}' y '{tipo_der}'."
            )

        tipo_resultante = "float" if ("float" in (tipo_izq, tipo_der)) else "int"

        if self.operador == "+":
            res = val_izq + val_der
        elif self.operador == "-":
            res = val_izq - val_der
        elif self.operador == "*":
            res = val_izq * val_der
        elif self.operador == "/":
            if val_der == 0:
                raise ZeroDivisionError("Error Semántico/Aritmético: División entre cero.")
            res = val_izq / val_der
            tipo_resultante = "float"
        else:
            raise ErrorSemantico(f"Error Semántico: Operador '{self.operador}' no soportado.")

        return res, tipo_resultante


class Declaracion(NodoAST):
    def __init__(self, nombre, tipo_declarado, expresion_inicial=None):
        self.nombre = nombre
        self.tipo_declarado = tipo_declarado
        self.expresion_inicial = expresion_inicial

    def evaluar(self, ts):
        val_expr = None
        if self.expresion_inicial is not None:
            # Evaluar expresión antes de registrar símbolo para resolver shadowing correctamente
            val_expr, tipo_expr = self.expresion_inicial.evaluar(ts)
            val_expr = validar_y_coercionar_asignacion(self.tipo_declarado, val_expr, tipo_expr, self.nombre)
        ts.declarar(self.nombre, self.tipo_declarado, val_expr)
        return val_expr, self.tipo_declarado


class Asignacion(NodoAST):
    def __init__(self, nombre_var, expresion):
        self.nombre_var = nombre_var
        self.expresion = expresion

    def evaluar(self, ts):
        simbolo = ts.buscar(self.nombre_var)
        val_expr, tipo_expr = self.expresion.evaluar(ts)
        val_expr = validar_y_coercionar_asignacion(simbolo.tipo_dato, val_expr, tipo_expr, self.nombre_var)
        simbolo.valor = val_expr
        print(f"   ✓ ASIGNACIÓN EXITOSA: {self.nombre_var} = {val_expr} (Tipo: {simbolo.tipo_dato})")
        return val_expr, simbolo.tipo_dato


class BloqueAST(NodoAST):
    def __init__(self, instrucciones, etiqueta=""):
        self.instrucciones = instrucciones
        self.etiqueta = etiqueta

    def evaluar(self, ts):
        ts.abrir_bloque(self.etiqueta)
        ultimo_res = None
        try:
            for instr in self.instrucciones:
                ultimo_res = instr.evaluar(ts)
        finally:
            # Garantiza la restauración del ámbito incluso ante excepciones semánticas
            ts.cerrar_bloque()
        return ultimo_res


# ------------------------------------------------------------------------------
# 4. DEMOSTRACIÓN DE EJECUCIÓN Y BATERÍA DE PRUEBAS SEMÁNTICAS
# ------------------------------------------------------------------------------

if __name__ == "__main__":
    ts = TablaSimbolosJerarquica()

    # Variables globales
    ts.declarar("global_a", "int", 10)
    ts.declarar("global_b", "float", 20.5)

    # Programa AST con anidamiento
    programa = BloqueAST(
        [
            Declaracion("x", "int", Operador("*", Variable("global_a"), Numero(2))),
            Asignacion("x", Operador("+", Variable("x"), Numero(5))),
            BloqueAST(
                [Declaracion("x", "float", Numero(3.14))],
                etiqueta="Bloque Anidado"
            ),
        ],
        etiqueta="Programa Principal"
    )

    print("--- Ejecutando Evaluación del AST e Integración con Tabla de Símbolos ---")
    programa.evaluar(ts)

    # Batería exhaustiva de pruebas semánticas
    print("\n--- Pruebas de Captura de Errores Semánticos ---")
    ts.declarar("texto", "string", "hola")
    ts.declarar("sin_valor", "int")

    casos_de_prueba = [
        ("A. Operación int + string", Operador("+", Variable("global_a"), Variable("texto"))),
        ("B. Inicializar float con string", Declaracion("f", "float", Variable("texto"))),
        ("C. Inicializar string con int", Declaracion("s", "string", Numero(5))),
        ("D. Variable no declarada", Variable("no_existe")),
        ("E. Variable sin inicializar", Variable("sin_valor")),
        ("F. Redeclaración en el mismo ámbito", Declaracion("global_a", "int", Numero(1))),
        ("G. Operador no soportado", Operador("%", Numero(5), Numero(2))),
        ("H. División entre cero", Operador("/", Numero(1), Numero(0))),
        ("I. Error dentro de bloque anidado (comprobando cierre seguro)",
         BloqueAST([Variable("no_existe")], etiqueta="Bloque con fallo")),
    ]

    for etiqueta, nodo in casos_de_prueba:
        print(f"\n   [Prueba {etiqueta}]")
        try:
            nodo.evaluar(ts)
            print("     ✗ Error no detectado")
        except (ErrorSemantico, ZeroDivisionError) as err:
            print(f"     [CAPTURADO ÉXITO]: {err}")

    print(f"\n   Ámbito activo final: Bloque {ts.ambito_actual.id_bloque} (Nivel {ts.ambito_actual.nivel})")

    # Demostración de coerciones numéricas
    print("\n--- Pruebas de Coerciones Numéricas Válidas ---")
    Declaracion("entero", "int", Numero(2.9)).evaluar(ts)
    Declaracion("real", "float", Variable("global_a")).evaluar(ts)
    print(f"   entero = {ts.buscar('entero').valor}, real = {ts.buscar('real').valor}")


   ✓ DECLARACIÓN: Variable 'global_a' (int) registrada en Bloque 0
   ✓ DECLARACIÓN: Variable 'global_b' (float) registrada en Bloque 0
--- Ejecutando Evaluación del AST e Integración con Tabla de Símbolos ---

[+] ABRIR BLOQUE 1 (Programa Principal) | Nivel 1
   ✓ DECLARACIÓN: Variable 'x' (int) registrada en Bloque 1
   ✓ ASIGNACIÓN EXITOSA: x = 25 (Tipo: int)

[+] ABRIR BLOQUE 2 (Bloque Anidado) | Nivel 2
   ✓ DECLARACIÓN: Variable 'x' (float) registrada en Bloque 2
[-] CERRAR BLOQUE 2 | Retornando a Nivel 1
[-] CERRAR BLOQUE 1 | Retornando a Nivel 0

--- Pruebas de Captura de Errores Semánticos ---
   ✓ DECLARACIÓN: Variable 'texto' (string) registrada en Bloque 0
   ✓ DECLARACIÓN: Variable 'sin_valor' (int) registrada en Bloque 0

   [Prueba A. Operación int + string]
     [CAPTURADO ÉXITO]: Error Semántico de Tipos: Operación '+' no permitida entre 'int' y 'string'.

   [Prueba B. Inicializar float con string]
     [CAPTURADO ÉXITO]: Error Semántico de Tipos: No se puede asignar 